In [1]:
import json

In [2]:
from camel_tools.disambig.mle import MLEDisambiguator
from camel_tools.tokenizers.word import simple_word_tokenize

model = MLEDisambiguator.pretrained()

In [3]:
def get_lemmas(sentence):
    disamb = model.disambiguate(simple_word_tokenize(sentence))

    results = {}
    for d in disamb:
        if len(d.analyses) > 0:
            results[d.word] =  d.analyses[0].analysis["lex"]
    return results
def line_lemmas(tokens):
        disamb = get_lemmas(" ".join(tokens))
        lemmas = []
        for token in tokens:
            if token in disamb:
                lemmas.append(disamb[token])
            else:
                lemmas.append(token)
        return lemmas

In [4]:
def format_token(token, state):
    switcher = {
        "corrupt": f"†{token}",
        "emended": f"*{token}",
        "unintelligible": f"?{token}",
        "lexical-error": f"!{token}",
        "dittography": f"[{token}]",
        "dittography_end": f"{token}]",
        "dittography_begin": f"[{token}",
        "cross-out": f"[[{token}]]",
        "cross-out_end": f"{token}]]",
        "cross-out_begin": f"[[{token}",
        "suppletion": f"{{{token}}}",
        "suppletion_end": token + "}",
        "suppletion_begin": "{" + token,
        "added": f"<{token}>",
        "added_end": f"{token}>",
        "added_begin": f"<{token}",
        "title": f"({token})",
        "title_end": f"{token})",
        "title_begin": f"({token}",
    }
    return switcher.get(state, token)

def format_token_rev(token, state):
    switcher = {
        "corrupt": f"{token}†",
        "emended": f"{token}*",
        "unintelligible": f"{token}?",
        "lexical-error": f"{token}!",
        "dittography": f"[{token}]",
        "dittography_end": f"[{token}",
        "dittography_begin": f"{token}]",
        "cross-out": f"[[{token}]]",
        "cross-out_end": f"[[{token}",
        "cross-out_begin": f"{token}]]",
        "suppletion": f"{{{token}}}",
        "suppletion_end": f"{{{token}",
        "suppletion_begin": f"{token}}}",
        "added": f"<{token}>",
        "added_end": f"<{token}",
        "added_begin": f"{token}>",
        "title": f"({token})",
        "title_end": f"({token}",
        "title_begin": f"{token})",
    }
    return switcher.get(state, token)


In [5]:
collations_and_orders = {
    "Mc": ["I344", "R2536", "P400", "P3465", "P5881", "P3475", "P3473", "R2407", "BWII672", "A4095", "P3466", "CCCP578", "L8751", "L4044", "P3471"],
    "Im": ["P3465", "P2789", "H170", "R2407", "P3473", "BWII672", "P3466", "T2281", "P3471", "P5881", "I344", "P400", "P3475", "A4095", "L8751", "L4044"],
    "Lv": ["P3465", "P3466", "P3473", "M487", "M618", "A4095", "P3475", "P400", "M486", "CCCP578", "P5881", "P3471"],
    "Oc": ["P3473", "BWII672", "P3465", "P3475", "A4095", "L8751", "L4044", "P3466", "P5881", "P400", "CCCP578", "R3655", "R2536", "P3471"],
    "Lj": ["P3475", "P3465", "P3473", "P400", "BWII672", "CCCP578", "P5881", "C66947", "CA11991", "A4095", "R2536", "P3471", "P3466", "Met-1981.373"],
    "Km": ["L8751", "A4095", "P3469", "P3468", "P3466", "VAF298"],
    "Kd": ["P3465", "P3473", "BWII672", "R2536", "I344", "P3471", "P400", "P3475", "CCCP578", "P5881", "A4095", "L8751", "B0022"],
    "Ag": ["P2789", "P3465", "M487", "P3466", "P3473", "A4095", "L8751", "P5881", "L4044"],
}
chapters_and_root_unit_titles = {
    "Mc": "Mouse and Cat",
    "Im": "Arabic Introduction",
    "Lv": "Burzoy's Voyage (long version)",
    "Oc": "Owls and Crows",
    "Lj": "Lion and Jackal",
    "Km": "The King of Mice",
    "Kd": "The King and 8 Dreams",
    "Ag": "Ascetic and Guest"
}
with open("manuscripts.json", "r", encoding="utf-8") as file:
    manuscripts_src = json.loads(file.read())

with open("units.json", "r", encoding="utf-8") as file:
    units_src = json.loads(file.read())
with open("segments.json", "r", encoding="utf-8") as file:
    segments_src = json.loads(file.read())
with open("pages.json", "r", encoding="utf-8") as file:
    pages_src = json.loads(file.read())
with open("text.json", "r", encoding="utf-8") as file:
    text_src = json.loads(file.read())
with open("lines.json", "r", encoding="utf-8") as file:
    lines_src = json.loads(file.read())
with open("images.json", "r", encoding="utf-8") as file:
    images_src = json.loads(file.read())

In [6]:
def get_mss_siglum_to_id(ms_list):
    res = {}
    for siglum in ms_list:
        for ms in manuscripts_src:
            if siglum == ms['siglum']:
                res[siglum] = ms['id']
    return res


In [7]:
def get_units(chapter):
    units = [x for x in units_src if x.get('frame', '') == chapter]
    unit_ids = {x['id'] for x in units}
    for unit in units:
        unit['longestSegment'] = {'count': 0, 'siglum': ''}
    return units, unit_ids

In [8]:
def format_units(units, root_unit_title):
    root_unit = next((x for x in units if x['title'] == root_unit_title), None)
    root_unit['formattedOrder'] = root_unit['frame']
    def get_childern(root_id, order):
        children = []
        for unit in units:
            if unit['parentId'] == root_id:
                unit['formattedOrder'] = f"{root_unit['frame']}.{unit['order']}"
                children.append(unit)
                children += get_childern(unit['id'], unit['order'])
        return children
    return root_unit, get_childern(root_unit['id'], f"{root_unit['order']}")

In [9]:
def get_segments(ms_id, unit_ids):
    segments = [x for x in segments_src if x['unitId'] in unit_ids and x['mediumId'] == ms_id]
    return {x['unitId']: x for x in segments}


In [10]:
def get_segment_pages(ms_id, segment):
    pages = [x for x in pages_src if x['mediumId'] == ms_id and x['number']>= segment['startPage'] and x['number'] <= segment['endPage']  ]
    return pages

def get_segment_bounding_pages(pages, segment):
    first = next((x for x in pages if x['number'] == segment['startPage']), None)
    last = next((x for x in pages if x['number'] == segment['endPage']), None)
    return first, last

In [11]:
def get_page_body_lines(page):
    element_ids = {x['id'] for x in text_src if x['pageId'] == page['id'] and 'main' in x['position']}
    lines = [x for x in lines_src if x['elementId'] in element_ids]
    for line in lines:
        line['pageNumber'] = page['number']
        line['pageId'] = page['id']
    return lines

In [12]:
def get_segment_lines(lines, segment, start_page_id, end_page_id):
    if segment["startPage"] == segment["endPage"]:
        return [
            x
            for x in lines
            if x["order"] >= segment["startLine"] and x["order"] <= segment["endLine"]
        ]
    if segment["endPage"] - segment["startPage"] == 1:
        lines_from_start_page = [
            x
            for x in lines
            if x["order"] >= segment["startLine"] and x["pageId"] == start_page_id
        ]
        lines_from_start_page.sort(key=lambda item: item["order"])
        lines_from_end_page = [
            x
            for x in lines
            if x["order"] <= segment["endLine"] and x["pageId"] == end_page_id
        ]
        lines_from_end_page.sort(key=lambda item: item["order"])
        return lines_from_start_page + lines_from_end_page
    if segment["endPage"] - segment["startPage"] > 1:
        lines_from_start_page = [
            x
            for x in lines
            if x["order"] >= segment["startLine"] and x["pageId"] == start_page_id
        ]
        lines_from_start_page.sort(key=lambda item: item["order"])
        lines_from_mid_pages = [
            x
            for x in lines
            if x["pageId"] != start_page_id and  x["pageId"] != end_page_id
        ]
        lines_from_mid_pages.sort(key=lambda item: (item["order"], item['pageNumber']))
        lines_from_end_page = [
            x
            for x in lines
            if x["order"] <= segment["endLine"] and x["pageId"] == end_page_id
        ]
        lines_from_end_page.sort(key=lambda item: item["order"])
        return lines_from_start_page + lines_from_mid_pages + lines_from_end_page

In [13]:
def get_line_data(line, start=None, end=None):
    if start is not None and end is not None:
        tokens = line['tokens'][start:end]
        states = line['states'][start:end]
    elif start is not None:
        tokens = line['tokens'][start:]
        states = line['states'][start:]
    elif end is not None:
        tokens = line['tokens'][:end]
        states = line['states'][:end]
    else:
        tokens = line['tokens']
        states = line['states']

    return {
        'tokens': [format_token(token, state) for token, state in zip(tokens, states)],
        'lemmas': line_lemmas(tokens),
        'lines': line['order'],
        'pages': line['pageNumber'],
        'breaks':  line['pageNumber'] if line['order'] == 0 and (start is None or start==0) else None,
    }

def get_segment_data_same_line(lines, segment):
    line_data = get_line_data(lines[0], start=segment['startToken'], end=segment['endToken']+1)
    data = {
        'images': [],
        'lemmas': [line_data['lemmas']],
        'tokens': [line_data['tokens']],
        'lines': [line_data['lines']],
        'pages': [line_data['pages']],
        'breaks': [line_data['breaks']],
    }
    return data



def get_segment_data_same_page(lines, segment):
    data = {
        'tokens': [],
        'lines': [],
        'pages': [],
        'images': [],
        'breaks': [],
        'lemmas': []
    }
    for line in lines:
        if line['order'] == segment['startLine']:
            line_data = get_line_data(line, start=segment['startToken'])
        elif line['order'] == segment['endLine']:
            line_data = get_line_data(line, end=segment['endToken']+1)
        else:
            line_data = get_line_data(line)

        data['tokens'].append(line_data['tokens'])
        data['lemmas'].append(line_data['lemmas'])
        data['lines'].append(line_data['lines'])
        data['pages'].append(line_data['pages'])
        data['breaks'].append(line_data['breaks'])

    return data


def get_segment_data_multiple_pages(lines, segment):
    data = {
        'tokens': [],
        'lines': [],
        'pages': [],
        'images': [],
        'breaks': [],
        'lemmas': []
    }
    for line in lines:
        if line['order'] == segment['startLine'] and line['pageNumber'] == segment['startPage']:
            line_data = get_line_data(line, start=segment['startToken'])
        elif line['order'] == segment['endLine'] and line['pageNumber'] == segment['endPage']:
            line_data = get_line_data(line, end=segment['endToken']+1)
        else:
            line_data = get_line_data(line)

        data['tokens'].append(line_data['tokens'])
        data['lemmas'].append(line_data['lemmas'])
        data['lines'].append(line_data['lines'])
        data['pages'].append(line_data['pages'])
        data['breaks'].append(line_data['breaks'])

    return data


def get_segment_data(lines, segment):
    if segment['startPage'] == segment['endPage']:
        if segment['startLine'] == segment['endLine']:
            return get_segment_data_same_line(lines, segment)
        else:
            return get_segment_data_same_page(lines, segment)
    else:
        return get_segment_data_multiple_pages(lines, segment)

def index_lemmas(segment_data, unit_index, column_index, lemmas, inverted_lemmas):
    lemmas[unit_index] = []
    for line_index, line in enumerate(segment_data['lemmas']):
        lemmas[unit_index].append(line)
        for lemma_index, lemma in enumerate(line):
            if lemma not in inverted_lemmas:
                inverted_lemmas[lemma] = []
            inverted_lemmas[lemma].append([unit_index, column_index, line_index, lemma_index])

    del segment_data['lemmas']
    pass

In [14]:
def pipeline(chapter, root_unit_title):
    mss_siglum_to_id = get_mss_siglum_to_id(collations_and_orders[chapter])
    units, unit_ids = get_units(chapter)
    root_unit, units = format_units(units, root_unit_title)
    units.sort(key=lambda item: item["order"])
    units = [root_unit] + units
    data = []
    lemmas = {}
    inverted_lemmas = {}
    for i, ms in enumerate(collations_and_orders[chapter]):
        id = mss_siglum_to_id[ms]
        segments = get_segments(mss_siglum_to_id[ms], unit_ids)
        dto = {"siglum": ms, "id": id, "order": i}
        ms_units = []
        facsimiles = {}
        lemmas[i] = {}
        for unit_index, unit in enumerate(units):
            id = unit["id"]
            if id not in segments or segments[id]["lacuna"] or unit["divider"]:
                ms_units.append(None)
            else:
                pages = get_segment_pages(mss_siglum_to_id[ms], segments[id])
                first, last = get_segment_bounding_pages(pages, segments[id])
                lines = []
                for page in pages:
                    facsimile = {"url": page["image"]}
                    page_lines = get_page_body_lines(page)
                    facsimile["lines"] = {x["order"]: x["region"] for x in page_lines}
                    facsimiles[page["number"]] = facsimile
                    lines += page_lines
                lines = get_segment_lines(lines, segments[id], first["id"], last["id"])
                segment_data = get_segment_data(
                    lines, segments[id]
                )  ## todo find what additional information should be includedin each segement dict
                index_lemmas(segment_data, unit_index, i, lemmas[i], inverted_lemmas)
                segments[id].update(segment_data)
                ms_units.append(segments[id])
                letters = " ".join( [token for line in segment_data["tokens"] for token in line])
                longest = max(unit["longestSegment"]["count"], len(letters))
                if longest == len(letters):
                    unit["longestSegment"] = {"count": longest, "siglum": ms}

        dto["ms_units"] = ms_units
        dto["facsimiles"] = facsimiles
        data.append(dto)
    for unit in units:
        unit["longestSegment"] = unit["longestSegment"]["siglum"]
    return units, data, lemmas, inverted_lemmas

In [15]:
def clone_dictionary(original_dict, excluded_fields):
    return {key: value for key, value in original_dict.items() if key not in excluded_fields}

def chuck_dictionary(data_dict):
    num_chunks = len(next(iter(data_dict.values()))) // 1  # Get length from the first list
    if len(next(iter(data_dict.values()))) % 1 != 0:
        num_chunks += 1

    for i in range(num_chunks):
        start = i * 1
        end = (i + 1) * 1
        chunk = {key: value[start:end][0] for key, value in data_dict.items()}  # Create chunk dictionary
        yield chunk

In [16]:
for CHAPTER in collations_and_orders.keys():
    units, data, lemmas, inverted_lemmas = pipeline(CHAPTER,  chapters_and_root_unit_titles[CHAPTER])
    OUT_PATH = f"../apps/data-api/data/collations/{CHAPTER}"
    with open(f"{OUT_PATH}/units.json", "w", encoding='utf-8') as write_file:
        json.dump(units, write_file)
    columns = []
    for dto in data:
        columns.append(clone_dictionary(dto, ['ms_units']))
    with open(f"{OUT_PATH}/columns.json", "w", encoding='utf-8') as write_file:
        json.dump(columns, write_file)
    segment_data = {x['siglum']: x['ms_units'] for x in data}
    segment_data = [x for x in chuck_dictionary(segment_data)]
    with open(f"{OUT_PATH}/segment_data.json", "w", encoding='utf-8') as write_file:
        json.dump(segment_data, write_file, ensure_ascii=False)
    with open(f"{OUT_PATH}/lemmas.json", "w", encoding='utf-8') as write_file:
        json.dump(lemmas, write_file, ensure_ascii=False)
    with open(f"{OUT_PATH}/inverted_lemmas.json", "w", encoding='utf-8') as write_file:
        json.dump(inverted_lemmas, write_file, ensure_ascii=False)
